# PAC v3.4.7 - Sistema Completo Trade Republic
**Alberto Scarin - PAC Puro Completo con Anti-Coltello - 11/04/2026**

### Struttura notebook:
| Cella | Funzione | Frequenza |
|---|---|---|
| 1 | Setup + Drive + flag RESET_ALL e RETRY_BLACKLIST | Ogni sessione |
| 2 | PDF TR + ISIN supplementari -> tr_raw_isin.csv | Solo se manca |
| 3 | ISIN -> Ticker Yahoo (OpenFIGI batch + blacklist) | Solo se manca |
| 3b | Retry blacklist + cleanup voci obsolete | Quando alzi RETRY_BLACKLIST |
| 4 | Download storici + volumi -> universe_static.pkl | Solo se manca |
| 5 | Filtro qualita + export + aggiorna blacklist | Solo se manca |
| **6** | **CANDIDATI settimanali** | **Ogni sabato** |
| 7 | Backtest storico completo | Su richiesta |

### File manuali su Drive (mai toccati dal sistema):
- `tr_universe.pdf` — PDF Trading Universe TR
- `isin_supplementari.csv` — ISIN aggiuntivi da fonti esterne
- `portafoglio_reale_input.csv` — Portafoglio attuale TR
- `pac_ok_flags.csv` — Flag PAC per asset (colonne: isin, pac_ok)

### Modifiche v3.4.7:
- `importlib` invece di `exec()` per caricamento helpers
- `except Exception as e` con logging invece di `except: pass` silenzioso
- `openfigi_batch` unica in helpers con retry 429
- Rimosso fallback ISIN come ticker Yahoo
- `itertuples()` invece di `iterrows()` nei loop
- Cleanup blacklist voci obsolete (> 12 mesi) in cella 3b


## Cella 1 - Setup
Esegui **sempre** all'inizio di ogni sessione Colab.

### Flag disponibili:
| Flag | Default | Effetto |
|---|---|---|
| `RESET_ALL` | `False` | Cancella tutti i file generati e riparte da zero |
| `RETRY_BLACKLIST` | `False` | Ri-testa ISIN in blacklist con retry_after scaduto + cleanup obsoleti |


In [ ]:
# CELLA 1 - Setup librerie + Google Drive + flag
# PAC v3.4.7

import subprocess
subprocess.run(["pip","install","pdfplumber","yfinance","pandas","numpy",
                "matplotlib","openpyxl","requests","tqdm","-q"], capture_output=True)

from google.colab import drive
import os, pickle, pandas as pd
from datetime import datetime, date
drive.mount("/content/drive", force_remount=False)

DRIVE_DIR = "/content/drive/MyDrive/PAC_v311"
os.makedirs(DRIVE_DIR, exist_ok=True)

RESET_ALL       = False
RETRY_BLACKLIST = False

# Configurazione periodo e modalita
PAC_START_YEAR = 2017
PAC_START_DATE = f"{PAC_START_YEAR}-01-01"  # download Yahoo (cella 4)
PAC_START_BT   = f"{PAC_START_YEAR}-03-01"  # backtest (3 mesi warmup EMA)
PAC_MODE       = "pure"  # "pure" = PAC classico | "timed" = market timing

PATHS = {
    "pdf_universe":    f"{DRIVE_DIR}/tr_universe.pdf",
    "supplementari":   f"{DRIVE_DIR}/isin_supplementari.csv",
    "portafoglio":     f"{DRIVE_DIR}/portafoglio_reale_input.csv",
    "pac_ok_flags":    f"{DRIVE_DIR}/pac_ok_flags.csv",
    "helpers":         f"{DRIVE_DIR}/pac_helpers.py",
    "raw_isin":        f"{DRIVE_DIR}/tr_raw_isin.csv",
    "ticker_map":      f"{DRIVE_DIR}/tr_isin_ticker.csv",
    "scartati":        f"{DRIVE_DIR}/tr_isin_scartati.csv",
    "universe_static": f"{DRIVE_DIR}/universe_static.pkl",
    "prices_weekly":   f"{DRIVE_DIR}/prices_weekly.pkl",
    "quality_csv":     f"{DRIVE_DIR}/tr_quality_tickers.csv",
    "quality_xlsx":    f"{DRIVE_DIR}/tr_quality_tickers.xlsx",
    "watchlist":       f"{DRIVE_DIR}/tr_watchlist_tradingview.csv",
    "top5_prev":       f"{DRIVE_DIR}/top5_prev.pkl",
    "backtest_weekly": f"{DRIVE_DIR}/backtest_weekly.csv",
    "backtest_moves":  f"{DRIVE_DIR}/backtest_movements.csv",
    "backtest_annual": f"{DRIVE_DIR}/backtest_annual.csv",
    "backtest_chart":  f"{DRIVE_DIR}/backtest_reale.png",
}
GENERATED_FILES = [
    "helpers","raw_isin","ticker_map","scartati","universe_static",
    "prices_weekly","quality_csv","quality_xlsx","watchlist",
    "top5_prev","backtest_weekly","backtest_moves","backtest_annual","backtest_chart",
]

if RESET_ALL:
    print("RESET_ALL = True -- cancellazione file generati...")
    deleted = 0
    for key in GENERATED_FILES:
        path = PATHS[key]
        if os.path.exists(path): os.remove(path); deleted += 1
    for f in os.listdir(DRIVE_DIR):
        if f.startswith("tr_sector_") and f.endswith(".csv"):
            os.remove(os.path.join(DRIVE_DIR, f)); deleted += 1
    print(f"OK: {deleted} file cancellati. File manuali intatti.")

# Scrivi helpers su Drive
HELPERS_SRC = '\nimport yfinance as yf\nimport pandas as pd\nimport requests\nimport time\nimport logging\n\n# Logger dedicato — visibile in Colab output\nlogger = logging.getLogger("pac_v331")\nif not logger.handlers:\n    handler = logging.StreamHandler()\n    handler.setFormatter(logging.Formatter("[PAC %(levelname)s] %(message)s"))\n    logger.addHandler(handler)\nlogger.setLevel(logging.WARNING)  # cambia a DEBUG per diagnostica\n\nimport re as _re_pac\n\n# ── COSTANTI OPENFIGI ─────────────────────────────────────────────\nFIGI_SUFFIX = {\n    "GY": ".DE", "LN": ".L",  "SM": ".MI",\n    "SS": ".ST", "FP": ".PA", "SW": ".SW",\n    "AU": ".AX", "HK": ".HK", "JP": ".T",\n}\nFIGI_PREFER_TYPES  = {"Common Stock", "ETP", "ETF", "Fund", "Mutual Fund"}\nFIGI_EUR_EXCHANGES = {"GY", "SM", "FP", "SS", "SW", "NA", "BB"}  # Xetra, Milan, Paris, Stockholm, Zurich, Amsterdam, Brussels\n\n# ── NORMALIZZAZIONE TICKER ────────────────────────────────────────\ndef normalize_ticker(ticker):\n    """strip + UPPERCASE — Yahoo e case-sensitive su alcuni exchange."""\n    if not ticker or not isinstance(ticker, str): return None\n    t = ticker.strip().upper()\n    return t if t else None\n\n_DIRTY_PAT = _re_pac.compile(r"[\\s/%()\\[\\]{}]|\\d{2}/\\d{2}/\\d{2,4}")\n\ndef is_valid_ticker(ticker):\n    """\n    True se il ticker e un simbolo Yahoo valido.\n    Esclude ticker con spazi, slash, date (es. obbligazioni OpenFIGI).\n    """\n    if not ticker: return False\n    if _DIRTY_PAT.search(ticker):\n        logger.debug(f"Ticker sporco escluso: {ticker!r}")\n        return False\n    return True\n\ndef dedupe_tickers(ok_map):\n    """\n    Rimuove collisioni ISIN -> stesso ticker Yahoo.\n    Casi: ETF share class diverse, ADR vs local listing.\n    """\n    seen = set(); clean = {}; dupes = []\n    for isin, row in ok_map.items():\n        t = row.get("ticker","")\n        if t in seen: dupes.append((isin,t)); continue\n        seen.add(t); clean[isin] = row\n    if dupes: logger.info(f"Ticker duplicati rimossi: {len(dupes)}")\n    return clean\n\nclass YfErrorType:\n    RATE_LIMIT = "rate_limit"\n    NO_DATA    = "no_data"\n    DELISTED   = "delisted"\n    UNKNOWN    = "unknown"\n\ndef classify_yf_error(err_str):\n    """Classifica errore Yahoo per gestione differenziata."""\n    e = err_str.lower()\n    if "429" in e or "rate" in e or "too many" in e: return YfErrorType.RATE_LIMIT\n    if "timeout" in e or "timed out" in e or "connection" in e or "ssl" in e:\n        return YfErrorType.RATE_LIMIT  # transiente: retry, non blacklistare\n    if "possibly delisted" in e or "no timezone" in e: return YfErrorType.DELISTED\n    if "no price data" in e or "prices missing" in e:  return YfErrorType.NO_DATA\n    return YfErrorType.UNKNOWN\n\n\n\n# ── YAHOO FINANCE CON RETRY ───────────────────────────────────────\ndef yf_download_safe(tickers, retries=3, pause=5, **kwargs):\n    """Download Yahoo Finance con retry su rate limit, timeout, SSL."""\n    for attempt in range(retries):\n        try:\n            raw = yf.download(tickers, progress=False, threads=True, **kwargs)\n            if raw is not None and not raw.empty:\n                return raw\n            time.sleep(pause)\n        except Exception as e:\n            etype = classify_yf_error(str(e))\n            if etype == YfErrorType.RATE_LIMIT:\n                wait = pause * (attempt + 1) * 5\n                logger.warning(f"Yahoo rate limit (attempt {attempt+1}/{retries}), attendo {wait}s")\n                time.sleep(wait)\n            elif etype in (YfErrorType.DELISTED, YfErrorType.NO_DATA):\n                logger.debug(f"Yahoo {etype}: {str(tickers)[:40]}")\n                return None  # inutile riprovare\n            else:\n                logger.warning(f"Yahoo errore (attempt {attempt+1}/{retries}): {e}")\n                time.sleep(pause * (attempt + 1))\n    logger.error(f"Yahoo download fallito dopo {retries} tentativi")\n    return None\n\n# ── OPENFIGI BATCH ────────────────────────────────────────────────\nFIGI_RATE_LIMIT = 3.0  # secondi tra batch (25 req/min)\n\ndef openfigi_batch(isin_list, retries=3):\n    """\n    Interroga OpenFIGI per un batch di ISIN (max 10 per chiamata).\n    Ritorna dict {isin: ticker_yahoo}.\n    Gestisce rate limit 429 con retry e backoff.\n    """\n    if not isin_list:\n        return {}\n    payload = [{"idType": "ID_ISIN", "idValue": isin} for isin in isin_list]\n    results = {}\n\n    for attempt in range(retries):\n        try:\n            resp = requests.post(\n                "https://api.openfigi.com/v3/mapping",\n                json=payload,\n                headers={"Content-Type": "application/json"},\n                timeout=10,\n            )\n            if resp.status_code == 429:\n                wait = 60 * (attempt + 1)\n                logger.warning(f"OpenFIGI rate limit 429, wait {wait}s (attempt {attempt+1})")\n                time.sleep(wait)\n                continue\n            if resp.status_code != 200:\n                logger.warning(f"OpenFIGI HTTP {resp.status_code} for batch {isin_list[:2]}")\n                return results\n\n            data = resp.json()\n            if not isinstance(data, list):\n                logger.warning(f"OpenFIGI unexpected response type: {type(data)}")\n                return results\n\n            for i, item in enumerate(data):\n                if i >= len(isin_list):\n                    break\n                isin = isin_list[i]\n                if not isinstance(item, dict) or "data" not in item:\n                    continue\n                candidates = item["data"]\n                if not candidates:\n                    continue\n                # Preferisci tipi azionari/ETF su exchange noti\n                ticker = None\n                # Pass 1: tipo preferito su exchange EUR\n                for cand in candidates:\n                    sec_type=(cand.get("securityType") or ""); tkr=(cand.get("ticker") or "").strip(); exch=(cand.get("exchCode") or "")\n                    if not tkr: continue\n                    t = normalize_ticker(tkr + FIGI_SUFFIX.get(exch,""))\n                    if not t or not is_valid_ticker(t): continue\n                    if exch in FIGI_EUR_EXCHANGES and any(pt.lower() in sec_type.lower() for pt in FIGI_PREFER_TYPES):\n                        ticker = t; break\n                # Pass 2: tipo preferito su qualsiasi exchange\n                if ticker is None:\n                    for cand in candidates:\n                        sec_type=(cand.get("securityType") or ""); tkr=(cand.get("ticker") or "").strip(); exch=(cand.get("exchCode") or "")\n                        if not tkr: continue\n                        t = normalize_ticker(tkr + FIGI_SUFFIX.get(exch,""))\n                        if not t or not is_valid_ticker(t): continue\n                        if any(pt.lower() in sec_type.lower() for pt in FIGI_PREFER_TYPES):\n                            ticker = t; break\n                # Fallback: primo risultato valido\n                if ticker:\n                    results[isin] = ticker\n            return results\n\n        except requests.exceptions.Timeout:\n            logger.warning(f"OpenFIGI timeout (attempt {attempt+1})")\n            time.sleep(5 * (attempt + 1))\n        except Exception as e:\n            logger.warning(f"OpenFIGI error (attempt {attempt+1}): {e}")\n            time.sleep(5)\n\n    logger.error(f"OpenFIGI failed after {retries} attempts for batch {isin_list[:2]}")\n    return results\n\n# ── VERIFICA STORICO YAHOO ────────────────────────────────────────\ndef verify_yahoo_ticker(ticker, min_weeks=26):\n    """\n    Verifica storico Yahoo. Usa ticker normalizzato (uppercase).\n    Ritorna: True | False | "rate_limit" (per gestione esterna).\n    """\n    t = normalize_ticker(ticker)\n    if not t or not is_valid_ticker(t):\n        return False\n    try:\n        raw = yf.download(t, period="1y", interval="1wk",\n                          auto_adjust=True, progress=False)\n        return raw is not None and not raw.empty and len(raw) >= min_weeks\n    except Exception as e:\n        etype = classify_yf_error(str(e))\n        if etype == YfErrorType.RATE_LIMIT:\n            logger.warning(f"verify_yahoo_ticker rate limit: {t}")\n            return "rate_limit"\n        logger.debug(f"verify_yahoo_ticker({t}): {e}")\n        return False\n\n# ── EMA ───────────────────────────────────────────────────────────\ndef compute_ema(series, period):\n    return series.ewm(span=period, adjust=False).mean()\n\n# ── SCORE v3.1.1 ─────────────────────────────────────────────────\ndef score_v311(c, date):\n    """Calcola score PAC v3.1.1 per un asset ad una data."""\n    try:\n        ps = c[c.index <= date]\n        if len(ps) < 4:\n            return None\n        if len(ps) > 1:\n            chg = ps.pct_change().abs()\n            ps  = ps[chg <= 0.80].copy()\n            if len(ps) < 4: return None\n        p0   = float(ps.iloc[-1])\n        jan1 = pd.Timestamp(date.year, 1, 1)\n        c_j  = ps[ps.index <= jan1]\n        c_1y = ps[ps.index <= date - pd.DateOffset(years=1)]\n        c_2y = ps[ps.index <= date - pd.DateOffset(years=2)]\n        if len(c_1y) == 0:\n            return None\n        p_jan = float(c_j.iloc[-1])  if len(c_j)  > 0 else float(c_1y.iloc[-1])\n        p1    = float(c_1y.iloc[-1])\n        p2    = float(c_2y.iloc[-1]) if len(c_2y) > 0 else p1\n        ytd = (p0 / p_jan - 1) * 100\n        r1  = (p0 / p1   - 1) * 100\n        r2  = (p0 / p2   - 1) * 100\n        def n(x): return max(0.0, min(100.0, (x + 60) / 180 * 100))\n        # Peso YTD: ridotto nei primi 60 giorni (dato troppo corto)\n        days_into_year = (date - pd.Timestamp(date.year, 1, 1)).days\n        if days_into_year < 60:\n            w_ytd, w_r1, w_r2 = 0.00, 0.60, 0.40\n        else:\n            w_ytd, w_r1, w_r2 = 0.30, 0.40, 0.30\n        ny, n1, n2 = n(ytd), n(r1), n(r2)\n        base = ny * w_ytd + n1 * w_r1 + n2 * w_r2\n        # Malus trend ribassista\n        if ytd < -15 and r1 < -10:\n            base *= 0.70\n        # Coefficiente persistenza: penalizza exploit mono-periodo\n        dispersion  = max(ny, n1, n2) - min(ny, n1, n2)\n        persistence = max(0.70, 1.0 - dispersion / 200.0)\n        base *= persistence\n        # Fix #2: anti-bolla — penalizza exploit YTD parabolici\n        # Micro-cap con YTD +150%+ che saturano lo score sono speculativi\n        if ytd > 300:\n            base = min(base, 60.0)  # cap duro: max 60 per exploit estremi\n        elif ytd > 150 and base >= 90:\n            base *= 0.60            # dimezza se YTD parabolico e score saturo\n        return round(base, 1)\n    except Exception as e:\n        logger.debug(f"score_v311 error: {e}")\n        return None\n\n# ── ANTI-COLTELLO ────────────────────────────────────────────────\ndef anti_coltello(c, date):\n    """\n    Moltiplicatore graduale per peso acquisto:\n      1.0  = sopra tutte le EMA (pieno acquisto)\n      0.75 = sotto EMA50 o EMA100, sopra EMA200 (acquisto ridotto)\n      0.5  = sotto EMA50 E EMA100, sopra EMA200 (meta quota)\n      0.0  = sotto EMA200 (blocco totale)\n    """\n    try:\n        ps   = c[c.index <= date]\n        if len(ps) < 10:\n            return 1.0  # dati insufficienti: non bloccare\n        e50  = float(compute_ema(ps, 50).iloc[-1])\n        e100 = float(compute_ema(ps, 100).iloc[-1])\n        e200 = float(compute_ema(ps, 200).iloc[-1])\n        p    = float(ps.iloc[-1])\n        if p < e200:\n            return 0.0   # blocco totale\n        if p < e50 and p < e100:\n            return 0.5   # sotto entrambe le EMA corte\n        if p < e50 or p < e100:\n            return 0.75  # sotto una sola EMA corta\n        return 1.0       # sopra tutto\n    except Exception as e:\n        logger.debug(f"anti_coltello: {e}")\n        return 1.0\n\n\n# ── FETCH SETTORE DA YAHOO FINANCE ───────────────────────────────\ndef fetch_yf_sector(ticker, retries=2):\n    """\n    Recupera sector e industry da yf.Ticker().info.\n    Ritorna (sector, industry) o (None, None) in caso di errore.\n    NON usare per ETF (info.sector spesso vuoto).\n    """\n    t = normalize_ticker(ticker)\n    if not t:\n        return None, None\n    for attempt in range(retries):\n        try:\n            info = yf.Ticker(t).info\n            sector   = info.get("sector")   or info.get("category") or None\n            industry = info.get("industry") or None\n            return sector, industry\n        except Exception as e:\n            etype = classify_yf_error(str(e))\n            if etype == YfErrorType.RATE_LIMIT:\n                logger.warning(f"fetch_yf_sector rate limit: {t}, wait 30s")\n                time.sleep(30)\n            else:\n                logger.debug(f"fetch_yf_sector({t}): {e}")\n                return None, None\n    return None, None\n\n# ── CLASSIFICAZIONE SETTORE ───────────────────────────────────────\ndef classify_sector(isin, name, yf_sector=None, yf_industry=None):\n    """\n    Classifica nel macro-settore PAC.\n    Priorita: yf_industry (GICS) > yf_sector > keyword > fallback geografico.\n    """\n    YF_INDUSTRY_MAP = {\n        "semiconductor":       "semiconductors",\n        "semiconductor equip": "semiconductors",\n        "electronic component":"semiconductors",\n        "gold":                "commodity",\n        "silver":              "commodity",\n        "copper":              "commodity",\n        "precious metal":      "commodity",\n        "other industrial met":"commodity",\n        "uranium":             "uranium",\n        "nuclear":             "uranium",\n        "aerospace & defense": "defense",\n        "drug manufactur":     "healthcare",\n        "biotechnology":       "healthcare",\n        "diagnostics":         "healthcare",\n        "medical device":      "healthcare",\n        "oil & gas":           "energy",\n        "banks":               "finance",\n        "asset management":    "finance",\n        "cryptocurrency":      "crypto",\n    }\n    YF_SECTOR_MAP = {\n        "technology":          "us_tech",\n        "communication":       "us_tech",\n        "healthcare":          "healthcare",\n        "energy":              "energy",\n        "basic materials":     "commodity",\n        "financial":           "finance",\n    }\n    # 1. Industry override (piu specifico)\n    if yf_industry and isinstance(yf_industry, str):\n        yi = yf_industry.lower()\n        for kw, macro in YF_INDUSTRY_MAP.items():\n            if kw in yi:\n                return macro\n    # 2. Sector map\n    if yf_sector and isinstance(yf_sector, str):\n        ys = yf_sector.lower()\n        for kw, macro in YF_SECTOR_MAP.items():\n            if kw in ys:\n                return macro\n    # 3. Fallback: keyword sul nome (ETF e asset senza .info)\n    n = name.lower()\n    i = isin.upper()\n    EXCLUDE_KW = [\n        "inverse","(-1x)","(-2x)","(-3x)","(2x) lev","(3x)","leveraged",\n        "short daily","daily short","daily (-","knock","turbo","warrant",\n        "bond","treasury","govt","government","sovereign","corp bond",\n        "inflation-linked","overnight return","yield plus","rente",\n        "daily (2","daily (3","2x lev","3x lev","factor leveraged",\n    ]\n    if any(kw in n for kw in EXCLUDE_KW): return "EXCLUDE"\n    if any(x in n for x in ["semiconductor","hynix","samsung electron","tsmc",\n        "taiwan semi","micron","asml","nvid","amd ","intel ","qualcomm",\n        "applied material","lam research","kla corp","advantest",\n        "tokyo electron","nxp ","broadcom"]): return "semiconductors"\n    if i[:9] in ["KR7000660","KR7005930"]: return "semiconductors"\n    if any(x in n for x in ["defense","defence","aerospace","rheinmetall",\n        "leonardo","bae system","lockheed","raytheon","northrop",\n        "general dynamics","kratos","airbus","thales","hanwha",\n        "saab ","kongsberg"]): return "defense"\n    if any(x in n for x in ["uranium","nuclear","cameco","paladin",\n        "boss energy","nexgen","sprott uranium","kazatom"]): return "uranium"\n    if any(x in n for x in ["physical gold","physical silver","gold eur",\n        "silver eur","gold usd","silver usd","gold miner","silver miner",\n        "newmont","barrick","agnico","kinross","wheaton","lithium","cobalt",\n        "copper miner","nickel ","rare earth","critical mineral",\n        "battery metal","remx","global x lithium",\n        "global x copper"]): return "commodity"\n    if any(x in n for x in ["oil & gas","stoxx oil","oilfield","schlumberger",\n        "halliburton","exxon","chevron","shell ","bp ","total energies",\n        "eni ","equinor","tanker","shipping","dry bulk"]): return "energy"\n    if any(x in n for x in ["health","pharma","biotech","medic","oncol",\n        "therapeut","lilly","novo nordisk","astrazeneca","pfizer","merck ",\n        "roche","novartis","gsk ","sanofi","abbott","vertex",\n        "regeneron"]): return "healthcare"\n    if any(x in n for x in ["nasdaq","information tech","s&p tech","software",\n        "cloud","cyber","artificial intel","machine learn","digital",\n        "apple ","microsoft","alphabet","meta ","amazon","salesforce",\n        "servicenow","palantir","crowdstrike","snowflake","datadog",\n        "fortinet"]): return "us_tech"\n    if any(x in n for x in ["s&p 500","s&p500","core s&p","msci world",\n        "ftse all-world","ftse all world","global equity","world equity",\n        "russell 2000","total market","world index"]): return "us_broad"\n    if any(x in n for x in ["china","chinese","hong kong","hang seng",\n        "csi 300","a-share","kweb","alibaba","tencent","baidu",\n        "pinduoduo","jd.com"]): return "china"\n    if any(x in n for x in ["india","indian","nifty","infosys","tata ",\n        "reliance","hdfc","vietnam","indonesia","thailand","malaysia",\n        "singapore","asean","japan","nikkei","topix","softbank","toyota",\n        "sony ","keyence","korea","kospi","hyundai","kakao","taiwan",\n        "mediatek","foxconn"]): return "em_asia"\n    if i[:2] in ["JP","KR","TW","SG","TH","ID","MY","VN"]: return "em_asia"\n    if any(x in n for x in ["brazil","mexico","latin","colombia","chile",\n        "peru","mercadolibre","nubank","itau","petrobras",\n        "vale "]): return "em_latam"\n    if any(x in n for x in ["emerging market","em equity"]): return "em_broad"\n    if any(x in n for x in ["bitcoin","crypto","blockchain","ethereum",\n        "coinbase","microstrategy","mara hold",\n        "riot platform"]): return "crypto"\n    if any(x in n for x in ["clean energy","renewable","solar","wind power",\n        "climate","robotics","automation","space ","drone",\n        "cybersecurity etf"]): return "thematic"\n    if any(x in n for x in ["poste italiane","ferrari","intesa","unicredit",\n        "enel ","eni ","leonardo","mediobanca",\n        "generali"]): return "italian"\n    if any(x in n for x in ["euro stoxx","stoxx europe","stoxx 600","stoxx 50",\n        "dax ","cac 40","ftse 100","msci europe","msci emu","msci germany",\n        "msci france","msci uk","rheinmetall","sap ","siemens","lvmh",\n        "nestle","unilever","volkswagen","bmw ","mercedes",\n        "abb "]): return "european"\n    if i[:2] in ["DE","FR","IT","ES","NL","SE","FI","DK","NO",\n                 "AT","BE","CH","PT","IE"]: return "european"\n    if i.startswith("AU"): return "australian"\n    if i.startswith("CA"): return "canadian"\n    # Mappatura industriale cross-border\n    # Intercetta asset classificati male dal fallback geografico\n    if any(x in n for x in ["gold","silver","copper","zinc","nickel",\n        "palladium","platinum"," mining"," miner","bullion",\n        "precious metal"," royalt"]):\n        return "commodity"\n    if any(x in n for x in ["semiconductor","chip","wafer","foundry",\n        "photonics","laser","optical","lidar","aehr","lumentum",\n        "ii-vi","coherent","ib photon"]):\n        return "semiconductors"\n    if any(x in n for x in ["biotech","genomic","oncology","therapeutic",\n        "biopharma","clinical stage","crispr","gene therapy"]):\n        return "healthcare"\n    if any(x in n for x in ["bank ","financ","jpmorgan","goldman sachs",\n        "bank of america","visa ","mastercard","blackrock","kkr ",\n        "blackstone"]): return "finance"\n    if i[:2] in ["SA","AE","QA","IL","ZA"]: return "middle_east"\n    return "other"\n\n# ── COSTANTI R2 ───────────────────────────────────────────────────\n\n# ── RSI + MARKET TIMING ─────────────────────────────────────────\ndef compute_rsi(series, period=14):\n    delta = series.diff()\n    gain  = delta.clip(lower=0).rolling(period).mean()\n    loss  = (-delta.clip(upper=0)).rolling(period).mean()\n    rs    = gain / loss.replace(0, float("nan"))\n    return 100 - (100 / (1 + rs))\n\ndef market_timing_multiplier(c, date):\n    """Moltiplicatore quota in PAC_MODE=timed."""\n    try:\n        ps = c[c.index <= date]\n        if len(ps) < 20: return 1.0\n        p    = float(ps.iloc[-1])\n        e200 = float(compute_ema(ps, 200).iloc[-1])\n        rsi  = float(compute_rsi(ps).iloc[-1])\n        ret4 = (p/float(ps.iloc[-5])-1)*100 if len(ps)>=5 else 0\n        near = 0 <= (p/e200-1) <= 0.02\n        if near and rsi < 35: return 1.8\n        if near:               return 1.5\n        if rsi < 35:           return 1.3\n        if rsi > 80:           return 0.1\n        if rsi > 75 and ret4 > 15: return 0.2\n        return 1.0\n    except Exception as e:\n        logger.debug(f"mt_mult: {e}"); return 1.0\n\ndef should_take_profit(c, date, avg_cost):\n    """True se rendimento>40% AND RSI>70 AND prezzo>EMA50*1.10."""\n    try:\n        ps  = c[c.index <= date]\n        if len(ps) < 20 or avg_cost <= 0: return False\n        p   = float(ps.iloc[-1])\n        e50 = float(compute_ema(ps, 50).iloc[-1])\n        rsi = float(compute_rsi(ps).iloc[-1])\n        return (p/avg_cost-1)*100 > 40 and rsi > 70 and p > e50*1.10\n    except Exception as e:\n        logger.debug(f"take_profit: {e}"); return False\n\nR2_LIMITS  = {"energy": 1, "crypto": 1, "thematic": 1, "bonds": 0, "EXCLUDE": 0}\nR2_DEFAULT = 2\n\n# ── IMPORTLIB LOADER ──────────────────────────────────────────────\ndef load_helpers(path):\n    """Carica helpers come modulo Python pulito (invece di exec)."""\n    import importlib.util\n    spec    = importlib.util.spec_from_file_location("pac_helpers", path)\n    module  = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    return module\n'
with open(PATHS["helpers"], "w") as f:
    f.write(HELPERS_SRC)

# Verifica scrittura riuscita prima di importare
_written = open(PATHS["helpers"]).read()
if len(_written) < 100:
    raise RuntimeError(f"pac_helpers.py scritto male ({len(_written)} bytes) -- riprova")

# Carica helpers come modulo pulito (importlib, non exec)
import importlib.util
_spec = importlib.util.spec_from_file_location("pac_helpers", PATHS["helpers"])
H = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(H)
# Esporta simboli nel namespace globale per compatibilita celle successive
yf_download_safe  = H.yf_download_safe
openfigi_batch    = H.openfigi_batch
verify_yahoo_ticker = H.verify_yahoo_ticker
dedupe_tickers      = H.dedupe_tickers
compute_ema                = H.compute_ema
compute_rsi                = H.compute_rsi
market_timing_multiplier   = H.market_timing_multiplier
should_take_profit         = H.should_take_profit
score_v311        = H.score_v311
anti_coltello     = H.anti_coltello
fetch_yf_sector   = H.fetch_yf_sector
classify_sector   = H.classify_sector
R2_LIMITS         = H.R2_LIMITS
R2_DEFAULT        = H.R2_DEFAULT
logger            = H.logger

print(f"PAC v3.4.7 -- {datetime.today().strftime('%d/%m/%Y %H:%M')}")
print("=" * 58)
checks = {
    "PDF Trading Universe TR":  PATHS["pdf_universe"],
    "ISIN supplementari":       PATHS["supplementari"],
    "Portafoglio reale":        PATHS["portafoglio"],
    "Flag pac_ok":              PATHS["pac_ok_flags"],
    "ISIN estratti":            PATHS["raw_isin"],
    "Ticker map (OpenFIGI)":    PATHS["ticker_map"],
    "Blacklist scartati":       PATHS["scartati"],
    "Universo statico":         PATHS["universe_static"],
    "Ticker CSV qualita":       PATHS["quality_csv"],
}
optional = {"ISIN supplementari","Portafoglio reale","Flag pac_ok"}
all_ready = True
for label, path in checks.items():
    exists = os.path.exists(path)
    size   = f"{os.path.getsize(path)//1024} KB" if exists else "--"
    icon   = "OK" if exists else "MANCA"
    if not exists and label not in optional: all_ready = False
    print(f"  {icon}  {label:<28} {size}")
if os.path.exists(PATHS["scartati"]):
    df_sc = pd.read_csv(PATHS["scartati"])
    today_str = date.today().isoformat()
    never_c   = (df_sc["retry_after"]=="never").sum() if "retry_after" in df_sc.columns else 0
    retry_due = sum(1 for ra in df_sc.get("retry_after",[])
                    if ra != "never" and isinstance(ra,str) and ra <= today_str)
    print(f"\n  Blacklist: {len(df_sc)} totali | {never_c} permanenti | {retry_due} pronti per retry")
print()
if all_ready:
    print("Sistema pronto -- esegui solo Cella 6 (CANDIDATI)")
    if RETRY_BLACKLIST:
        print("RETRY_BLACKLIST = True -- esegui Cella 3b")
else:
    print("File mancanti -- esegui le celle 2->5 in sequenza")
print(f"Helpers caricati: {[x for x in dir(H) if not x.startswith('_')]}")


## Cella 2 - Estrazione ISIN
Estrae ISIN dal PDF TR e unisce con ISIN supplementari.

**Output:** `tr_raw_isin.csv`

Salta se il file esiste gia su Drive.


In [ ]:
# CELLA 2 - Estrazione ISIN dal PDF TR + merge ISIN supplementari
import pdfplumber, re, os, pandas as pd

if os.path.exists(PATHS["raw_isin"]):
    df_raw = pd.read_csv(PATHS["raw_isin"])
    print(f"OK: ISIN gia su Drive: {len(df_raw)} -- cella saltata")
else:
    if not os.path.exists(PATHS["pdf_universe"]):
        raise FileNotFoundError(
            "tr_universe.pdf non trovato su Drive.\n"
            "Caricalo in MyDrive/PAC_v311/"
        )
    print("Estrazione ISIN dal PDF Trade Republic...")
    # Pattern 1: ISIN a inizio riga + nome (formato standard TR)
    ISIN_LINE_RE = re.compile(r"^([A-Z]{2}[A-Z0-9]{10})\s+(.+)$")
    # Pattern 2: ISIN ovunque nella riga (PDF a colonne o layout diverso)
    ISIN_ANY_RE  = re.compile(r"\b([A-Z]{2}[A-Z0-9]{10})\b")
    instruments = {}
    with pdfplumber.open(PATHS["pdf_universe"]) as pdf:
        for page in pdf.pages:
            text  = page.extract_text() or ""
            for line in text.split("\n"):
                s = line.strip()
                m = ISIN_LINE_RE.match(s)
                if m:
                    isin = m.group(1); name = m.group(2).strip()
                    if isin not in instruments: instruments[isin] = name
                    continue
                # Fallback: cerca ISIN ovunque nella riga
                for isin in ISIN_ANY_RE.findall(s):
                    if isin not in instruments:
                        rest = s[s.find(isin)+12:].strip()
                        # Sanity check: scarta se testo e sporco
                        # (numeri isolati, codici brevi = layout PDF a colonne)
                        import re as _re
                        rest_clean = _re.sub(r"[\d.,%-]+", "", rest).strip()
                        if len(rest_clean) < 3:
                            rest = isin  # usa ISIN come nome placeholder
                        instruments[isin] = rest
    n_pdf = len(instruments)
    print(f"  ISIN dal PDF TR: {n_pdf}")
    if n_pdf == 0:
        print("  ATTENZIONE: zero ISIN estratti dal PDF.")
        print("  Verifica che sia il PDF Trading Universe TR (non protetto o scansionato).")
        print("  Procedo comunque con i soli ISIN supplementari se presenti.")
    if os.path.exists(PATHS["supplementari"]):
        try:
            df_supp = pd.read_csv(PATHS["supplementari"])
            df_supp.columns = [c.lower() for c in df_supp.columns]
            if "isin" in df_supp.columns and "name" in df_supp.columns:
                added = sum(
                    1 for row in df_supp.itertuples()
                    if len(str(row.isin).strip()) == 12
                    and str(row.isin).strip() not in instruments
                    and not instruments.update({str(row.isin).strip(): str(row.name).strip()})
                )
                print(f"  ISIN supplementari aggiunti: {added}")
        except Exception as e:
            print(f"  ATTENZIONE: errore lettura isin_supplementari.csv: {e}")
    df_raw = pd.DataFrame(list(instruments.items()), columns=["isin","name"])
    df_raw.to_csv(PATHS["raw_isin"], index=False)
    print(f"\nOK: {len(df_raw)} ISIN totali salvati su Drive")


## Cella 3 - Mappa ISIN -> Ticker Yahoo (OpenFIGI)

### Logica blacklist:
| Condizione | Azione |
|---|---|
| ISIN gia in `tr_isin_ticker.csv` | Saltato |
| ISIN in blacklist `never` | Saltato permanentemente |
| ISIN in blacklist con `retry_after` futuro | Saltato |
| ISIN nuovo o `retry_after` scaduto | Interrogato su OpenFIGI |

### Motivi di scarto:
| Motivo | retry_after |
|---|---|
| OpenFIGI: ticker non trovato | oggi + 90 giorni |
| Yahoo: storico < 26 settimane | oggi + 90 giorni |
| EXCLUDE (turbo/short/bond) | `never` |

**Nota:** il ticker fallback ISIN->Yahoo e stato rimosso. Solo ticker verificati da OpenFIGI vengono usati.

Tempo stimato prima esecuzione: **~90 minuti** (batch da 10 + fetch `.info` Yahoo per ogni ticker)

**Colonne prodotte in `tr_isin_ticker.csv`:**
`isin | name | ticker | source | yf_sector | yf_industry`


In [ ]:
# CELLA 3 - Mappa ISIN -> Ticker Yahoo (OpenFIGI batch + blacklist)
# Input:  tr_raw_isin.csv, tr_isin_scartati.csv
# Output: tr_isin_ticker.csv, tr_isin_scartati.csv (aggiornato)
import pandas as pd, os, time
from datetime import date, timedelta

MIN_WEEKS_VERIFY = 26
BATCH_SIZE_FIGI  = 10
CHECKPOINT_EVERY = 100
TODAY_STR = date.today().isoformat()

df_raw = pd.read_csv(PATHS["raw_isin"])
all_raw_isins = set(df_raw["isin"])

# Carica stato attuale (checkpoint o vuoto)
ok_map = {}
if os.path.exists(PATHS["ticker_map"]):
    for row in pd.read_csv(PATHS["ticker_map"]).itertuples(index=False):
        ok_map[row.isin] = {"isin":row.isin,"name":row.name,
                            "ticker":row.ticker,"source":getattr(row,"source","")}
sc_map = {}
if os.path.exists(PATHS["scartati"]):
    for row in pd.read_csv(PATHS["scartati"]).itertuples(index=False):
        sc_map[row.isin] = dict(row._asdict())

# Ticker pre-noti da supplementari (top-level: sempre disponibile)
supp_tickers = {}
if os.path.exists(PATHS["supplementari"]):
    try:
        df_s = pd.read_csv(PATHS["supplementari"])
        df_s.columns = [c.lower() for c in df_s.columns]
        tkr_col = next((c for c in df_s.columns
                        if c in ("ticker","ticker_yahoo","symbol","yf_ticker")), None)
        # Fallback: usa la terza colonna se esiste (indice 2)
        if tkr_col is None and len(df_s.columns) >= 3:
            tkr_col = df_s.columns[2]
        if tkr_col and "isin" in df_s.columns:
            for row in df_s.itertuples(index=False):
                t = str(getattr(row, tkr_col, "")).strip()
                if t and t.lower() not in ("nan","none",""):
                    supp_tickers[str(row.isin).strip()] = t
            if supp_tickers:
                print("  Ticker pre-noti da supplementari: " + str(len(supp_tickers)))
    except Exception as e:
        logger.warning("supp_tickers: " + str(e))

# Verifica completezza reale
processed     = set(ok_map.keys()) | set(sc_map.keys())
remaining_all = all_raw_isins - processed

if not remaining_all:
    print(f"OK: tutti i {len(all_raw_isins)} ISIN elaborati -- cella saltata")
    print(f"    Ticker: {len(ok_map)} | Blacklist: {len(sc_map)}")
else:
    if processed:
        print(f"Ripresa da checkpoint: {len(processed)} elaborati, {len(remaining_all)} rimanenti")
    else:
        print(f"Primo avvio: {len(df_raw)} ISIN da elaborare")

    # Costruisci lista da interrogare
    to_process = []
    skip_ok = skip_never = skip_future = 0
    for row in df_raw.itertuples(index=False):
        isin = row.isin; name = row.name
        if isin in ok_map:  skip_ok += 1; continue
        if isin in sc_map:
            ra = sc_map[isin].get("retry_after","never")
            if ra == "never": skip_never += 1; continue
            if ra > TODAY_STR: skip_future += 1; continue
        to_process.append((isin, name))

    print(f"  Gia elaborati:       {skip_ok + skip_never + skip_future}")
    print(f"  Da interrogare OFIGI:{len(to_process)}")
    est = max(1, len(to_process) // BATCH_SIZE_FIGI) * H.FIGI_RATE_LIMIT / 60
    print(f"  Tempo stimato:       ~{est:.0f} minuti")

    processed_count = 0
    for batch_start in range(0, len(to_process), BATCH_SIZE_FIGI):
        batch       = to_process[batch_start:batch_start+BATCH_SIZE_FIGI]
        batch_isins = [b[0] for b in batch]
        batch_names = {b[0]: b[1] for b in batch}

        # ISIN con ticker gia noto da supplementari: salta OpenFIGI
        need_figi   = [i for i in batch_isins if i not in supp_tickers]
        skip_figi   = [i for i in batch_isins if i in supp_tickers]
        ticker_results = openfigi_batch(need_figi) if need_figi else {}
        # Aggiungi i ticker pre-noti al risultato
        for i in skip_figi: ticker_results[i] = supp_tickers[i]
        if need_figi: time.sleep(H.FIGI_RATE_LIMIT)

        for isin in batch_isins:
            name   = batch_names[isin]
            sector = classify_sector(isin, name)
            ticker = ticker_results.get(isin)  # None se non trovato

            if sector == "EXCLUDE":
                sc_map[isin] = {"isin":isin,"name":name,"motivo":"exclude_structure",
                                "fase":"3","data_scarto":TODAY_STR,"retry_after":"never"}
                continue

            # Nessun fallback ISIN->ticker: se OpenFIGI non trova, e scarto
            vfy = verify_yahoo_ticker(ticker) if ticker else False
            if vfy == "rate_limit":
                logger.warning(f"Rate limit Yahoo su {ticker}, ISIN {isin} saltato")
            elif vfy is True:
                # Fetch settore/industry da Yahoo .info (una tantum)
                yf_sec, yf_ind = fetch_yf_sector(ticker)
                ok_map[isin] = {"isin":isin,"name":name,"ticker":ticker,
                                "source":"openfigi",
                                "yf_sector":yf_sec or "","yf_industry":yf_ind or ""}
                time.sleep(0.5)  # piccola pausa per non stressare Yahoo .info
            else:
                motivo = "openfigi_no_ticker" if not ticker else "yahoo_no_history"
                retry  = (date.today() + timedelta(days=90)).isoformat()
                sc_map[isin] = {"isin":isin,"name":name,"motivo":motivo,
                                "fase":"3","data_scarto":TODAY_STR,"retry_after":retry}

        processed_count += len(batch)
        if processed_count % CHECKPOINT_EVERY == 0 or \
           batch_start + BATCH_SIZE_FIGI >= len(to_process):
            pd.DataFrame(list(ok_map.values())).to_csv(PATHS["ticker_map"], index=False)
            pd.DataFrame(list(sc_map.values())).to_csv(PATHS["scartati"], index=False)
            pct = processed_count / len(to_process) * 100 if to_process else 100
            print(f"  Checkpoint {processed_count}/{len(to_process)} ({pct:.0f}%)"
                  f" -- OK: {len(ok_map)} | Scartati: {len(sc_map)}")

    ok_map    = dedupe_tickers(ok_map)  # rimuove collisioni ISIN->stesso ticker
    df_ok     = pd.DataFrame(list(ok_map.values()))
    df_sc_out = pd.DataFrame(list(sc_map.values()))
    df_ok.to_csv(PATHS["ticker_map"], index=False)
    df_sc_out.to_csv(PATHS["scartati"], index=False)
    never_c = (df_sc_out["retry_after"]=="never").sum() if len(df_sc_out) else 0
    print(f"\nOK: {len(df_ok)} ticker verificati")
    print(f"    Blacklist: {len(df_sc_out)} totali ({never_c} permanenti)")


## Cella 3b - Retry Blacklist + Cleanup

**Esegui solo quando `RETRY_BLACKLIST = True` in cella 1.**

### Cosa fa:
1. **Cleanup**: rimuove voci con `retry_after` scaduto da oltre 12 mesi (irrecuperabili)
2. **Retry**: ri-testa ISIN con `retry_after` <= oggi
3. Chi passa -> torna in `tr_isin_ticker.csv`
4. Chi fallisce -> aggiorna `retry_after` a oggi + 90 giorni

**`openfigi_batch` e `verify_yahoo_ticker` sono definite una sola volta in helpers.**


In [ ]:
# CELLA 3b - Retry blacklist + cleanup voci obsolete
import pandas as pd, os, time
from datetime import date, timedelta

if not RETRY_BLACKLIST:
    print("RETRY_BLACKLIST = False -- cella saltata")
    print("Imposta RETRY_BLACKLIST = True in cella 1 per eseguire")
else:
    TODAY      = date.today()
    TODAY_STR  = TODAY.isoformat()
    CUTOFF_STR = (TODAY - timedelta(days=365)).isoformat()  # cleanup > 12 mesi

    if not os.path.exists(PATHS["scartati"]):
        print("Blacklist non trovata -- nulla da fare")
    else:
        df_sc = pd.read_csv(PATHS["scartati"])
        print(f"Blacklist attuale: {len(df_sc)} voci")

        # Step 1: Cleanup voci obsolete (non-never, retry_after < cutoff 12 mesi fa)
        def is_obsolete(row):
            ra = row.get("retry_after","never")
            if ra == "never": return False
            return isinstance(ra, str) and ra < CUTOFF_STR
        obsolete_mask = df_sc.apply(is_obsolete, axis=1)
        n_obsolete    = obsolete_mask.sum()
        df_sc_clean   = df_sc[~obsolete_mask].copy()
        print(f"  Cleanup voci obsolete (> 12 mesi): {n_obsolete} rimosse")

        # Step 2: Identifica candidati per retry
        eligible_mask = (
            (df_sc_clean["retry_after"] != "never") &
            (df_sc_clean["retry_after"] <= TODAY_STR)
        )
        eligible   = df_sc_clean[eligible_mask].copy()
        ineligible = df_sc_clean[~eligible_mask].copy()
        print(f"  Permanenti (never):  {(df_sc_clean['retry_after']=='never').sum()}")
        print(f"  Pronti per retry:    {len(eligible)}")
        print(f"  Retry futuro:        {(~eligible_mask & (df_sc_clean['retry_after']!='never')).sum()}")

        if len(eligible) == 0:
            print("\nNessun ISIN da ri-testare oggi.")
            df_sc_clean.to_csv(PATHS["scartati"], index=False)
            print(f"OK: blacklist aggiornata ({len(df_sc_clean)} voci dopo cleanup)")
        else:
            # Carica ticker map esistente
            ok_map = {}
            if os.path.exists(PATHS["ticker_map"]):
                for row in pd.read_csv(PATHS["ticker_map"]).itertuples(index=False):
                    ok_map[row.isin] = dict(row._asdict())

            promoted = []; still_failing = []

            for batch_start in range(0, len(eligible), 10):
                batch       = eligible.iloc[batch_start:batch_start+10]
                batch_isins = list(batch["isin"])
                batch_names = dict(zip(batch["isin"], batch["name"]))

                ticker_results = openfigi_batch(batch_isins)  # unica implementazione da helpers
                time.sleep(H.FIGI_RATE_LIMIT)

                for isin in batch_isins:
                    ticker = ticker_results.get(isin)
                    if ticker and verify_yahoo_ticker(ticker):
                        ok_map[isin] = {"isin":isin,"name":batch_names[isin],
                                        "ticker":ticker,"source":"retry"}
                        promoted.append(isin)
                    else:
                        still_failing.append(isin)

            # Ricostruisce blacklist:
            # - rimuove promossi
            # - aggiorna retry_after per i falliti
            # - mantiene ineligible invariati
            promoted_set = set(promoted)
            new_sc_rows  = []
            for row in ineligible.itertuples(index=False):
                new_sc_rows.append(dict(row._asdict()))
            for row in eligible.itertuples(index=False):
                if row.isin in promoted_set: continue
                d = dict(row._asdict())
                if row.isin in still_failing:
                    d["retry_after"] = (TODAY + timedelta(days=90)).isoformat()
                new_sc_rows.append(d)

            pd.DataFrame(list(ok_map.values())).to_csv(PATHS["ticker_map"], index=False)
            pd.DataFrame(new_sc_rows).to_csv(PATHS["scartati"], index=False)

            print(f"\n" + "="*50)
            print(f"RETRY COMPLETATO")
            print(f"  Testati:        {len(eligible)}")
            print(f"  Promossi a OK:  {len(promoted)}")
            print(f"  Ancora falliti: {len(still_failing)}")
            print(f"  Blacklist dopo: {len(new_sc_rows)} voci")
            if promoted:
                print(f"\nISIN promossi:")
                for isin in promoted:
                    print(f"  {isin}  {ok_map[isin]['name'][:40]}  -> {ok_map[isin]['ticker']}")
            print("\nRicorda: imposta RETRY_BLACKLIST = False in cella 1")


## Cella 4 - Download storici

### Logica blacklist:
| Motivo | retry_after |
|---|---|
| Download Yahoo fallisce | oggi + 90 giorni |
| Storico < 3 anni | oggi + 365 giorni |

Riprende dal checkpoint se interrotta. ISIN in blacklist saltati senza download.


In [ ]:
# CELLA 4 - Download storici + volumi -> universe_static.pkl
# Input:  tr_isin_ticker.csv, tr_isin_scartati.csv
# Output: universe_static.pkl, tr_isin_scartati.csv (aggiornato)
import pandas as pd, pickle, os, time
from datetime import date, timedelta

START_DATE = PAC_START_DATE  # da cella 1
MIN_WEEKS  = 156
BATCH_SIZE = 50
SAVE_EVERY = 500
TODAY_STR  = date.today().isoformat()

if not os.path.exists(PATHS["ticker_map"]):
    raise FileNotFoundError("tr_isin_ticker.csv non trovato. Esegui prima la cella 3.")

df_map = pd.read_csv(PATHS["ticker_map"])
# Solo righe con ticker valido (non ISIN come fallback)
df_map = df_map[df_map["ticker"].notna() & (df_map["ticker"].str.strip() != "")].copy()
isin_to_ticker   = dict(zip(df_map["isin"], df_map["ticker"].str.strip()))
isin_to_name     = dict(zip(df_map["isin"], df_map["name"]))
isin_to_yf_sec   = dict(zip(df_map["isin"], df_map.get("yf_sector",  pd.Series(dtype=str)).fillna("")))
isin_to_yf_ind   = dict(zip(df_map["isin"], df_map.get("yf_industry", pd.Series(dtype=str)).fillna("")))

sc_map = {}
if os.path.exists(PATHS["scartati"]):
    for row in pd.read_csv(PATHS["scartati"]).itertuples(index=False):
        sc_map[row.isin] = dict(row._asdict())

# Filtra ISIN: salta blacklist permanenti e non scaduti
all_isins = []
skip_bl   = 0
for isin in df_map["isin"]:
    if isin in sc_map:
        ra = sc_map[isin].get("retry_after","never")
        if ra == "never" or ra > TODAY_STR: skip_bl += 1; continue
    all_isins.append(isin)
print(f"ISIN da processare: {len(all_isins)} (saltati blacklist: {skip_bl})")

if os.path.exists(PATHS["universe_static"]):
    with open(PATHS["universe_static"],"rb") as f: universe = pickle.load(f)
    prices = universe.get("prices",{})
    print(f"Cache esistente: {len(prices)} ISIN")
else:
    prices = {}

# Verifica completezza reale
sc_set  = set(sc_map.keys())
missing = [i for i in all_isins if i not in prices and i not in sc_set]
if not missing:
    print(f"OK: tutti i {len(all_isins)} ISIN elaborati -- salto download")
elif prices or sc_set:
    done = len(set(prices.keys()) | (sc_set & set(all_isins)))
    print(f"Ripresa da checkpoint: {done}/{len(all_isins)} elaborati, {len(missing)} rimanenti")
print(f"Da scaricare: {len(missing)} ISIN")

new_failures = []
for idx in range(0, len(missing), BATCH_SIZE):
    batch_isins = missing[idx:idx+BATCH_SIZE]
    # Solo ticker verificati — nessun fallback ISIN
    batch_items   = [(i, isin_to_ticker[i]) for i in batch_isins if i in isin_to_ticker]
    if not batch_items: continue
    batch_isins_f = [x[0] for x in batch_items]
    batch_tickers = [x[1] for x in batch_items]

    raw = yf_download_safe(batch_tickers, start=START_DATE, interval="1wk", auto_adjust=True)
    if raw is None or raw.empty:
        for isin in batch_isins_f:
            retry = (date.today()+timedelta(days=90)).isoformat()
            sc_map[isin] = {"isin":isin,"name":isin_to_name.get(isin,""),
                            "motivo":"yahoo_download_failed","fase":"4",
                            "data_scarto":TODAY_STR,"retry_after":retry}
            new_failures.append(isin)
        continue

    is_multi = isinstance(raw.columns, pd.MultiIndex)
    has_close = "Close" in (raw.columns.get_level_values(0) if is_multi else raw.columns)
    has_vol   = "Volume" in (raw.columns.get_level_values(0) if is_multi else raw.columns)
    if not has_close: continue

    for isin, ticker in batch_items:
        try:
            c = raw["Close"][ticker].dropna() if is_multi and ticker in raw["Close"].columns \
                else raw["Close"].dropna() if not is_multi else pd.Series(dtype=float)
            v = raw["Volume"][ticker].dropna() if is_multi and has_vol and ticker in raw["Volume"].columns \
                else raw["Volume"].dropna() if not is_multi and has_vol else pd.Series(dtype=float)
            if len(c) >= MIN_WEEKS:
                prices[isin] = {
                    "close":  c, "volume": v,
                    "sector": classify_sector(isin, isin_to_name.get(isin,""),
                              isin_to_yf_sec.get(isin,""),
                              isin_to_yf_ind.get(isin,"")),
                    "name":   isin_to_name.get(isin,""),
                    "ticker": ticker,
                }
            else:
                retry = (date.today()+timedelta(days=365)).isoformat()
                sc_map[isin] = {"isin":isin,"name":isin_to_name.get(isin,""),
                                "motivo":"insufficient_history","fase":"4",
                                "data_scarto":TODAY_STR,"retry_after":retry}
                new_failures.append(isin)
        except Exception as e:
            logger.warning(f"Cella 4: errore ISIN {isin}/{ticker}: {e}")

    if (idx+BATCH_SIZE) % SAVE_EVERY == 0:
        with open(PATHS["universe_static"],"wb") as f:
            pickle.dump({"prices":prices,"isin_to_name":isin_to_name},f)
        pd.DataFrame(list(sc_map.values())).to_csv(PATHS["scartati"],index=False)
        pct = min(100,(idx+BATCH_SIZE)/len(missing)*100) if missing else 100
        print(f"  {idx+BATCH_SIZE}/{len(missing)} ({pct:.0f}%) -- in cache: {len(prices)}")
    time.sleep(0.2)

# Aggiornamento delta prezzi recenti
if all_isins:
    print("Aggiornamento delta prezzi recenti...")
    cached_items = [(i,isin_to_ticker[i]) for i in all_isins if i in prices and i in isin_to_ticker]
    for idx in range(0, len(cached_items), BATCH_SIZE*2):
        batch = cached_items[idx:idx+BATCH_SIZE*2]
        batch_is = [x[0] for x in batch]; batch_tk = [x[1] for x in batch]
        raw = yf_download_safe(batch_tk, period="1mo", interval="1wk", auto_adjust=True)
        if raw is None or raw.empty: continue
        is_multi = isinstance(raw.columns, pd.MultiIndex)
        for isin, ticker in batch:
            try:
                new_c = raw["Close"][ticker].dropna() if is_multi and ticker in raw["Close"].columns \
                        else raw["Close"].dropna() if not is_multi and "Close" in raw.columns \
                        else pd.Series(dtype=float)
                if len(new_c) == 0: continue
                merged = pd.concat([prices[isin]["close"], new_c])
                prices[isin]["close"] = merged[~merged.index.duplicated(keep="last")].sort_index()
            except Exception as e:
                logger.debug(f"Delta update {isin}: {e}")
        time.sleep(0.1)

with open(PATHS["universe_static"],"wb") as f:
    pickle.dump({"prices":prices,"isin_to_name":isin_to_name},f)
pd.DataFrame(list(sc_map.values())).to_csv(PATHS["scartati"],index=False)
size_mb = os.path.getsize(PATHS["universe_static"])//1024//1024
print(f"\nOK: universe_static.pkl -- {len(prices)} ISIN ({size_mb} MB)")
if new_failures: print(f"    Aggiunti a blacklist: {len(new_failures)} ISIN")


## Cella 5 - Filtro qualita + export + aggiorna blacklist

### Motivi di scarto e retry_after:
| Motivo | retry_after |
|---|---|
| EXCLUDE (turbo/short/bond) | `never` |
| Prezzo < EUR 3 | oggi + 180 giorni |
| Turnover < EUR 500k/sett | oggi + 90 giorni |
| Dati < 3 anni | oggi + 365 giorni |
| Return 1Y > 400% anomalo | oggi + 90 giorni |

### File esportati:
- `tr_quality_tickers.csv` — lista con metriche e `pac_ok`
- `tr_quality_tickers.xlsx` — Excel (Tutti + PAC_OK + per settore)
- `tr_watchlist_tradingview.csv` — watchlist TradingView
- `tr_sector_<settore>.csv` — per settore


In [ ]:
# CELLA 5 - Filtro qualita + export + aggiorna blacklist
import pandas as pd, pickle, os
from datetime import date, timedelta

TODAY_STR = date.today().isoformat()

if os.path.exists(PATHS["quality_csv"]):
    df_clean = pd.read_csv(PATHS["quality_csv"])
    print(f"OK: Universo qualita gia su Drive: {len(df_clean)} ticker")
    print(f"    PAC-abili: {df_clean['pac_ok'].sum()}")
else:
    with open(PATHS["universe_static"],"rb") as f: universe = pickle.load(f)
    prices = universe["prices"]
    sc_map = {}
    if os.path.exists(PATHS["scartati"]):
        for row in pd.read_csv(PATHS["scartati"]).itertuples(index=False):
            sc_map[row.isin] = dict(row._asdict())
    pac_flags = {}
    if os.path.exists(PATHS["pac_ok_flags"]):
        try:
            df_flags = pd.read_csv(PATHS["pac_ok_flags"])
            if {"isin","pac_ok"}.issubset(set(df_flags.columns)):
                pac_flags = dict(zip(df_flags["isin"],df_flags["pac_ok"].astype(bool)))
        except Exception as e:
            print(f"ATTENZIONE: errore lettura pac_ok_flags: {e}")

    MIN_PRICE=3.0; MIN_WEEKS=156; MAX_1Y_RET=400; MIN_TURNOVER=500_000
    print(f"Filtro qualita su {len(prices)} ISIN...")
    results=[]; new_scraps=0

    for isin, data in prices.items():
        sector = data.get("sector","other")
        name   = data.get("name","")
        # Se sector e "other" prova a riclassificare con dati yf
        if sector == "other":
            sector = classify_sector(isin, name,
                                     data.get("yf_sector",""),
                                     data.get("yf_industry",""))
        if sector == "EXCLUDE":
            if isin not in sc_map or sc_map[isin].get("retry_after") != "never":
                sc_map[isin] = {"isin":isin,"name":name,"motivo":"exclude_structure",
                                "fase":"5","data_scarto":TODAY_STR,"retry_after":"never"}
                new_scraps += 1
            continue
        c = data["close"]; v = data.get("volume", pd.Series(dtype=float))
        if len(c) < MIN_WEEKS:
            if isin not in sc_map:
                sc_map[isin] = {"isin":isin,"name":name,"motivo":"insufficient_history_quality",
                                "fase":"5","data_scarto":TODAY_STR,
                                "retry_after":(date.today()+timedelta(days=365)).isoformat()}
                new_scraps += 1
            continue
        price = float(c.iloc[-1])
        if price < MIN_PRICE:
            if isin not in sc_map:
                sc_map[isin] = {"isin":isin,"name":name,"motivo":"price_too_low",
                                "fase":"5","data_scarto":TODAY_STR,
                                "retry_after":(date.today()+timedelta(days=180)).isoformat()}
                new_scraps += 1
            continue
        p1y   = float(c.iloc[-53]) if len(c)>53 else float(c.iloc[0])
        ret1y = (price/p1y-1)*100 if p1y>0 else 0
        if ret1y > MAX_1Y_RET:
            if isin not in sc_map:
                sc_map[isin] = {"isin":isin,"name":name,"motivo":"anomalous_return",
                                "fase":"5","data_scarto":TODAY_STR,
                                "retry_after":(date.today()+timedelta(days=90)).isoformat()}
                new_scraps += 1
            continue
        p2y   = float(c.iloc[-105]) if len(c)>105 else float(c.iloc[0])
        ret2y = (price/p2y-1)*100 if p2y>0 else 0
        jan1  = pd.Timestamp(c.index[-1].year, 1, 1)
        c_jan = c[c.index <= jan1]
        p_jan = float(c_jan.iloc[-1]) if len(c_jan)>0 else p1y
        ytd   = (price/p_jan-1)*100 if p_jan>0 else 0
        turnover = 0
        if len(v) > 4:
            turnover = float(v.tail(8).mean()) * price
        if turnover > 0 and turnover < MIN_TURNOVER:
            if isin not in sc_map:
                sc_map[isin] = {"isin":isin,"name":name,"motivo":"low_turnover",
                                "fase":"5","data_scarto":TODAY_STR,
                                "retry_after":(date.today()+timedelta(days=90)).isoformat()}
                new_scraps += 1
            continue
        # Fix #5: filtro giorni piatti — asset con >30% settimane a var. zero
        # Sintomo di illiquidita strutturale non catturata dal turnover
        if len(c) > 10:
            flat_pct = float((c.pct_change().abs() < 0.001).mean())
            if flat_pct > 0.30:
                if isin not in sc_map:
                    sc_map[isin] = {"isin":isin,"name":name,"motivo":"illiquid_flat_days",
                                    "fase":"5","data_scarto":TODAY_STR,
                                    "retry_after":(date.today()+timedelta(days=90)).isoformat()}
                    new_scraps += 1
                continue
        results.append({
            "isin":isin,"name":name,"ticker":data.get("ticker",isin),
            "sector":sector,"country":isin[:2],"weeks":len(c),
            "price_eur":round(price,2),"ytd_pct":round(ytd,1),
            "ret_1y_pct":round(ret1y,1),"ret_2y_pct":round(ret2y,1),
            "turnover_weekly_eur":int(turnover) if turnover>0 else None,
            "pac_ok":bool(pac_flags.get(isin,True)),
        })

    df_clean = pd.DataFrame(results).sort_values(
        "turnover_weekly_eur", ascending=False, na_position="last")
    pd.DataFrame(list(sc_map.values())).to_csv(PATHS["scartati"],index=False)

    print(f"\nUNIVERSO TR PULITO: {len(df_clean)} ticker")
    print(f"  PAC-abili:              {df_clean['pac_ok'].sum()}")
    print(f"  Nuovi scarti blacklist: {new_scraps}")
    for sec, cnt in df_clean["sector"].value_counts().items():
        pac_c = int(df_clean[df_clean["sector"]==sec]["pac_ok"].sum())
        print(f"  {sec:<20} {cnt:>5}  (PAC-OK: {pac_c})")

    df_clean.to_csv(PATHS["quality_csv"],index=False)
    df_clean[["isin","name","ticker","sector","pac_ok"]].to_csv(PATHS["watchlist"],index=False)
    with pd.ExcelWriter(PATHS["quality_xlsx"]) as writer:
        df_clean.to_excel(writer,sheet_name="Tutti",index=False)
        df_clean[df_clean["pac_ok"]].to_excel(writer,sheet_name="PAC_OK",index=False)
        for sec in df_clean["sector"].unique():
            df_clean[df_clean["sector"]==sec].to_excel(writer,sheet_name=sec[:31],index=False)
    for sec in df_clean["sector"].unique():
        df_clean[df_clean["sector"]==sec].to_csv(
            f"{DRIVE_DIR}/tr_sector_{sec}.csv",index=False)
    df_bl = pd.DataFrame(list(sc_map.values()))
    never_c = (df_bl["retry_after"]=="never").sum() if len(df_bl) else 0
    print(f"\nBlacklist: {len(df_bl)} totali ({never_c} permanenti)")
    print("OK: file esportati su Drive")


## Cella 6 - CANDIDATI Settimanali
**Esegui ogni sabato dopo la cella 1.**

### Regole pac_ok:
| Stato | Comportamento |
|---|---|
| `pac_ok=True`, non in portafoglio | Elaborato, puo entrare nel top-5 |
| `pac_ok=True`, in portafoglio | Elaborato + monitorato |
| `pac_ok=False`, non in portafoglio | **Ignorato completamente** |
| `pac_ok=False`, in portafoglio congelato | **Solo monitoraggio pareggio** |

### Output:
- **TOP-5** con ISIN, settore, score, EUR/settimana
- **5 ALTERNATIVI** se un top-5 non e attivabile su TR
- **Congelati pac_ok=False** con stato pareggio
- **TOP-25** per score

### Limiti R2:
| Settore | Max slot | Motivo |
|---|---|---|
| energy | 1 | Rischio geopolitico |
| crypto | 1 | Volatilita estrema |
| thematic | 1 | Speculativi |
| tutti gli altri | 2 | — |


In [ ]:
# CELLA 6 - CANDIDATI SETTIMANALI PAC v3.4.7
import pandas as pd, pickle, os
from datetime import datetime

TODAY    = datetime.today()
DATE_STR = TODAY.strftime("%d/%m/%Y %H:%M")
print(f"PAC v3.4.7 [{PAC_MODE.upper()}] -- {DATE_STR}\n")

with open(PATHS["universe_static"],"rb") as f: universe = pickle.load(f)
prices_static = universe["prices"]
print(f"Universo caricato: {len(prices_static)} ISIN")

pac_flags = {}
if os.path.exists(PATHS["pac_ok_flags"]):
    try:
        df_flags = pd.read_csv(PATHS["pac_ok_flags"])
        if {"isin","pac_ok"}.issubset(set(df_flags.columns)):
            pac_flags = dict(zip(df_flags["isin"],df_flags["pac_ok"].astype(bool)))
    except Exception as e:
        logger.warning(f"pac_ok_flags: {e}")

portafoglio = {}
if os.path.exists(PATHS["portafoglio"]):
    try:
        df_ptf = pd.read_csv(PATHS["portafoglio"])
        if {"isin","nome","valore_eur","pl_acquisto_eur"}.issubset(set(df_ptf.columns)):
            for row in df_ptf.itertuples(index=False):
                portafoglio[row.isin] = {
                    "nome":  row.nome,
                    "valore":float(row.valore_eur),
                    "costo": float(row.valore_eur) - float(row.pl_acquisto_eur),
                }
            print(f"Portafoglio: {len(portafoglio)} posizioni")
    except Exception as e:
        logger.warning(f"portafoglio_reale: {e}")

print("Aggiornamento prezzi recenti...")
prices = {k: dict(v) for k,v in prices_static.items()}
all_isins   = list(prices.keys())
all_tickers = [prices[i].get("ticker") for i in all_isins]
# Filtra coppie con ticker valido
valid_pairs = [(i,t) for i,t in zip(all_isins,all_tickers) if t and isinstance(t,str) and t.strip()]
BATCH=100; updated=0
for idx in range(0, len(valid_pairs), BATCH):
    batch   = valid_pairs[idx:idx+BATCH]
    batch_is = [x[0] for x in batch]; batch_tk = [x[1] for x in batch]
    raw = yf_download_safe(batch_tk, period="1mo", interval="1wk", auto_adjust=True)
    if raw is None or raw.empty: continue
    is_multi = isinstance(raw.columns, pd.MultiIndex)
    for isin, ticker in batch:
        try:
            new_c = raw["Close"][ticker].dropna() if is_multi and ticker in raw["Close"].columns \
                    else raw["Close"].dropna() if not is_multi and "Close" in raw.columns \
                    else pd.Series(dtype=float)
            if len(new_c) == 0: continue
            merged = pd.concat([prices[isin]["close"], new_c])
            prices[isin]["close"] = merged[~merged.index.duplicated(keep="last")].sort_index()
            updated += 1
        except Exception as e:
            logger.debug(f"Aggiornamento {isin}: {e}")
with open(PATHS["prices_weekly"],"wb") as f: pickle.dump(prices,f)
print(f"OK: {updated} prezzi aggiornati")
# Fix #6: segnala asset senza dati recenti (sospetto delisted)
sospetti = []
for isin in prices:
    cl = prices[isin]["close"]
    if len(cl) == 0: continue
    days_old = (pd.Timestamp(TODAY) - cl.index[-1]).days
    if days_old > 21:
        sospetti.append((isin, prices[isin].get("name",""), prices[isin].get("ticker",""), days_old))
if sospetti:
    n_s = len(sospetti)
    print("")
    print("SUSPETTO DELISTED: " + str(n_s) + " asset senza dati da oltre 3 settimane")
    for isin,name,tkr,d in sorted(sospetti,key=lambda x:-x[3])[:10]:
        print("  " + isin.ljust(14) + tkr.ljust(12) + name[:28] + " (" + str(d) + "gg fa)")

print("Calcolo score v3.1.1...")
date_now   = pd.Timestamp(TODAY)
scores     = {}
sector_map = {i: prices[i].get("sector","other") for i in prices}
name_map   = {i: prices[i].get("name","")        for i in prices}

for isin in prices:
    sector    = sector_map.get(isin,"other")
    if sector == "EXCLUDE": continue
    pac_ok    = bool(pac_flags.get(isin, True))
    is_in_ptf = isin in portafoglio
    if not pac_ok and not is_in_ptf: continue
    if not pac_ok and is_in_ptf:     continue
    s = score_v311(prices[isin]["close"], date_now)
    if s is not None: scores[isin] = s
print(f"Asset con score: {len(scores)}")

top5_list=[]; alt_list=[]; macro_cnt={}; blocked_r2=[]; blocked_ac=[]
for isin, score in sorted(scores.items(), key=lambda x: -x[1]):
    sector = sector_map.get(isin,"other")
    limit  = R2_LIMITS.get(sector, R2_DEFAULT)
    ac_w   = anti_coltello(prices[isin]["close"], date_now)  # 0.0/0.5/0.75/1.0
    eff_sc = round(score * ac_w, 1)  # score effettivo dopo Anti-Coltello
    if limit == 0 or macro_cnt.get(sector,0) >= limit:
        blocked_r2.append((isin,score,sector,name_map.get(isin,"")))
        if ac_w > 0.0 and len(alt_list) < 5:
            alt_list.append((isin,eff_sc,sector,name_map.get(isin,""),ac_w))
        continue
    if ac_w == 0.0:
        blocked_ac.append((isin,score,sector,name_map.get(isin,""))); continue
    if len(top5_list) < 5:
        top5_list.append((isin,eff_sc,sector,name_map.get(isin,""),ac_w))
        macro_cnt[sector] = macro_cnt.get(sector,0) + 1
    elif len(alt_list) < 5:
        alt_list.append((isin,eff_sc,sector,name_map.get(isin,""),ac_w))
    if len(top5_list) == 5 and len(alt_list) == 5: break

tot_sc    = sum(s for isin,s,*_ in top5_list) or 1
q_raw     = {isin: s/tot_sc*100 for isin,s,*_ in top5_list}
if PAC_MODE == "timed":
    mt_m  = {isin: market_timing_multiplier(prices[isin]["close"], date_now) for isin in q_raw}
    tot_a = sum(q_raw[i]*mt_m[i] for i in q_raw) or 1
    quotes = {isin: int(q_raw[isin]*mt_m[isin]/tot_a*100) for isin in q_raw}
else:
    quotes = {isin: int(v) for isin,v in q_raw.items()}
if top5_list: quotes[top5_list[0][0]] += 100 - sum(quotes.values())

print(f"\n{'='*75}")
print(f"CANDIDATI TOP-5 -- {DATE_STR}")
print(f"{'='*75}")
print(f"{'#':<3} {'ISIN':<14} {'Settore':<16} {'Score':>6} {'EUR/sett':>9}  Nome")
print("-"*75)
for i,item in enumerate(top5_list,1):
    isin,eff_sc,sector,name,ac_w = item
    raw_sc = scores.get(isin, eff_sc)
    ac_tag = "" if ac_w==1.0 else f" [AC:{ac_w:.2f}x]"
    print(f"{i:<3} {isin:<14} {sector:<16} {raw_sc:>6.1f} {quotes[isin]:>8}   {name[:28]}{ac_tag}")
if not top5_list: print("  Nessun candidato -- verifica connessione")
print(f"{'':>60} {'100 EUR':>9}")
print(f"{'='*75}")

if alt_list:
    print(f"\nALTERNATIVI (attiva se un top-5 non e PAC-able su TR):")
    print(f"{'#':<4} {'ISIN':<14} {'Settore':<16} {'Score':>6}  Nome")
    print("-"*65)
    for i,item in enumerate(alt_list,6):
        isin,eff_sc,sector,name,ac_w = item
        raw_sc = scores.get(isin, eff_sc)
        ac_tag = "" if ac_w==1.0 else f" [AC:{ac_w:.2f}x]"
        print(f"{i:<4} {isin:<14} {sector:<16} {raw_sc:>6.1f}  {name[:28]}{ac_tag}")

congelati_nopac = [
    (isin,d["nome"],d["valore"],d["costo"],
     "PAREGGIO RAGGIUNTO" if d["valore"]>=d["costo"]
     else f"{(d['valore']/d['costo']-1)*100:+.1f}%")
    for isin,d in portafoglio.items() if not pac_flags.get(isin,True)
]
if congelati_nopac:
    print(f"\nCONGELATI (pac_ok=False) -- monitoraggio pareggio:")
    print(f"{'ISIN':<14} {'Nome':<30} {'Valore':>10} {'Costo':>10} {'Stato':>20}")
    print("-"*88)
    for isin,nome,val,costo,stato in congelati_nopac:
        print(f"{isin:<14} {nome:<30} {val:>9.2f}   {costo:>9.2f}   {stato:>20}")

if blocked_r2:
    print(f"\nBLOCCATI R2:")
    for isin,s,sec,name in blocked_r2[:6]:
        print(f"   {isin:<14} {sec:<16} {s:.1f}  {name[:30]}  (max {R2_LIMITS.get(sec,R2_DEFAULT)})")
if blocked_ac:
    print(f"\nBLOCCATI Anti-Coltello:")
    for isin,s,sec,name in blocked_ac[:5]:
        print(f"   {isin:<14} {sec:<16} {s:.1f}  {name[:30]}")

top5_set = {item[0] for item in top5_list}
alt_set  = {item[0] for item in alt_list}
r2_set   = {item[0] for item in blocked_r2}
ac_set   = {item[0] for item in blocked_ac}
print(f"\nTOP-25 per score:")
print(f"{'#':>3} {'ISIN':<14} {'Settore':<16} {'Score':>6} {'AC':>4} {'Stato':<14} Nome")
print("-"*80)
cnt=0
for isin,s in sorted(scores.items(),key=lambda x:-x[1]):
    if cnt>=25: break
    sec=sector_map.get(isin,"?"); name=name_map.get(isin,"")
    ac_w_v = anti_coltello(prices[isin]["close"],date_now)
    ac="100%" if ac_w_v==1.0 else (f"{int(ac_w_v*100)}%" if ac_w_v>0 else "BLOK")
    if   isin in top5_set: stato="TOP-5"
    elif isin in alt_set:  stato="ALTERNATIVO"
    elif isin in r2_set:   stato="R2"
    elif isin in ac_set:   stato="AC"
    else:                  stato="--"
    print(f"{cnt+1:>3}. {isin:<14} {sec:<16} {s:>6.1f} {ac:>5} {stato:<14} {name[:25]}")
    cnt+=1

print(f"\nLimiti R2:")
for sec,lim in sorted(R2_LIMITS.items()):
    if sec in ("bonds","EXCLUDE"): continue
    print(f"   {sec:<16} max {lim}{' <- ridotto' if lim<2 else ''}")
print(f"   {'altri settori':<16} max 2")

with open(PATHS["top5_prev"],"wb") as f:
    pickle.dump({"date":DATE_STR,"top5":top5_list,"alt":alt_list},f)
print(f"\nOK: top5_prev.pkl salvato su Drive")


## Cella 7 - Backtest storico (su richiesta)

### Fix v3.4.7:
- Filtro data quotazione: asset non ancora esistenti alla data simulata vengono saltati
- Metriche avanzate: Max Drawdown, Calmar ratio, Sharpe ratio, anni positivi/negativi
- Grafico doppio: valore vs versato + drawdown nel tempo

**Output:** `backtest_weekly.csv`, `backtest_movements.csv`, `backtest_annual.csv`, `backtest_reale.png`


In [ ]:
# CELLA 7 - Backtest storico completo PAC v3.4.7
import pandas as pd, numpy as np, pickle, os
import matplotlib.pyplot as plt, matplotlib.ticker as mticker
from datetime import datetime
exec(open(PATHS["helpers"]).read())

with open(PATHS["universe_static"],"rb") as f: universe=pickle.load(f)
prices_base=universe["prices"]
if os.path.exists(PATHS["prices_weekly"]):
    with open(PATHS["prices_weekly"],"rb") as f: pw=pickle.load(f)
    prices={**prices_base,**pw}
else: prices=prices_base
sector_map={i:prices[i].get("sector","other") for i in prices}
name_map={i:prices[i].get("name","") for i in prices}
print(f"Asset: {len(prices)}")
BUDGET=100.0; COMM=1.0; INFL=3.56
def select_top5_bt(sc_dict,prices_dict,date):
    ranked=sorted(sc_dict.items(),key=lambda x:-x[1]); top5=[]; mc={}
    for isin,score in ranked:
        sec=sector_map.get(isin,"other")
        if sec=="EXCLUDE": continue
        limit=R2_LIMITS.get(sec,R2_DEFAULT)
        if limit==0 or mc.get(sec,0)>=limit: continue
        if not anti_coltello(prices_dict[isin]["close"],date): continue
        top5.append((isin,score)); mc[sec]=mc.get(sec,0)+1
        if len(top5)==5: break
    return top5
all_dates=pd.date_range(start=PAC_START_BT,end=pd.Timestamp.today(),freq="W-MON")
print(f"Settimane: {len(all_dates)}")
shares={}; cost={}; frozen=set(); capitale_ext=0.0; cassa=0.0
prev_top5=[]; log=[]; events=[]; weekly_rows=[]; move_rows=[]
for i,date in enumerate(all_dates):
    if i%52==0: print(f"  {date.strftime('%Y-%m')} -- {i}/{len(all_dates)}")
    cp={}
    for isin,data in prices.items():
        c=data["close"]
        # Escludi asset non ancora esistenti alla data simulata
        if len(c) == 0 or c.index[0] > date: continue
        cn=c[c.index<=date]
        if len(cn)>0:
            try: cp[isin]=float(cn.iloc[-1])
            except: pass
    if not cp: continue
    sc={}
    for isin in cp:
        # Escludi asset non ancora quotati alla data simulata
        first_date = prices[isin]["close"].index[0]
        if first_date > date: continue
        s=score_v311(prices[isin]["close"],date)
        if s is not None: sc[isin]=s
    if not sc: continue
    top5_list=select_top5_bt(sc,prices,date); top5=[isin for isin,_ in top5_list]
    for isin in list(frozen):
        if isin in cp and shares.get(isin,0)>0.001 and cost.get(isin,0)>0:
            val=shares[isin]*cp[isin]
            if val>=cost[isin]:
                prov=val-COMM; cassa+=max(prov,0); pl=val-cost[isin]
                events.append({"tipo":"PAREGGIO","data":str(date.date()),"asset":name_map.get(isin,isin),"pl":round(pl,1)})
                move_rows.append({"data":str(date.date()),"tipo":"VENDI_PAREGGIO","isin":isin,"nome":name_map.get(isin,isin),"importo":round(val,2),"pl":round(pl,2)})
                shares[isin]=0.0; cost[isin]=0.0; frozen.discard(isin)
    for isin in list(frozen):
        if isin in top5: frozen.discard(isin)
    for isin in list(shares.keys()):
        if shares.get(isin,0)>0.001 and isin not in top5 and isin not in frozen:
            if isin in prev_top5: frozen.add(isin)
    disponibile=BUDGET+cassa; capitale_ext+=BUDGET; cassa=0.0
    tot_sc=sum(s for _,s in top5_list) or 1
    for isin,s in top5_list:
        if isin not in cp: continue
        q_base = disponibile*s/tot_sc
        if PAC_MODE == "timed":
            shs = shares.get(isin,0); cst = cost.get(isin,0)
            avg_c = cst/shs if shs > 0.001 else 0
            if avg_c > 0 and should_take_profit(prices[isin]["close"], date, avg_c):
                sell = shs*0.5*cp[isin] - COMM
                if sell > 0:
                    cassa += sell; cost[isin]=cst*0.5; shares[isin]=shs*0.5
                    move_rows.append({"data":str(date.date()),"tipo":"TAKEPROFIT",
                        "isin":isin,"nome":name_map.get(isin,isin),"importo":round(sell,2),"pl":round(sell-cst*0.5,2)})
            q_base *= market_timing_multiplier(prices[isin]["close"], date)
        q = max(0, q_base)
        if q < 0.01: continue
        shares[isin]=shares.get(isin,0)+q/cp[isin]; cost[isin]=cost.get(isin,0)+q
        move_rows.append({"data":str(date.date()),"tipo":"ACQUISTO","isin":isin,"nome":name_map.get(isin,isin),"importo":round(q,2),"pl":0})
    vt=sum(shares[isin]*cp[isin] for isin in shares if shares.get(isin,0)>0.001 and isin in cp)
    weekly_rows.append({"data":str(date.date()),"investito":round(capitale_ext),"valore":round(vt),"pl":round(vt-capitale_ext),"top5":" | ".join(name_map.get(i,i)[:10] for i,_ in top5_list)})
    if i%4==0:
        pl=vt-capitale_ext
        log.append({"date":str(date.date()),"year":date.year,"inv":round(capitale_ext),"val":round(vt),"pl":round(pl),"plp":round(pl/capitale_ext*100,1) if capitale_ext else 0,"top5":[name_map.get(i,i)[:12] for i,_ in top5_list]})
    prev_top5=top5[:]
last_date=all_dates[-1]
cp_fin={}
for isin,data in prices.items():
    c=data["close"]; cn=c[c.index<=last_date]
    if len(cn)>0:
        try: cp_fin[isin]=float(cn.iloc[-1])
        except: pass
vf=sum(shares[isin]*cp_fin[isin] for isin in shares if shares.get(isin,0)>0.001 and isin in cp_fin)
pl_f=vf-capitale_ext; plp=pl_f/capitale_ext*100 if capitale_ext else 0
n_years=(last_date-all_dates[0]).days/365.25
cagr=((vf/capitale_ext)**(1/n_years)-1)*100 if capitale_ext>0 else 0
reale=((1+cagr/100)/(1+INFL/100)-1)*100
# Calcolo metriche avanzate
# Max Drawdown
vals_series = [l["val"] for l in log]
peak = vals_series[0]; max_dd = 0.0; dd_start = 0; dd_end = 0; dd_peak_idx = 0
peak_idx = 0
for idx_v, v in enumerate(vals_series):
    if v > peak:
        peak = v; peak_idx = idx_v
    dd = (peak - v) / peak * 100 if peak > 0 else 0
    if dd > max_dd:
        max_dd = dd; dd_start = peak_idx; dd_end = idx_v
# Durata drawdown in settimane (ogni punto log = 4 settimane)
dd_weeks = (dd_end - dd_start) * 4
# Recovery: trova quando il valore torna sopra il picco dopo dd_end
recovery_weeks = None
peak_val_at_dd = vals_series[dd_start] if dd_start < len(vals_series) else 0
for idx_v in range(dd_end, len(vals_series)):
    if vals_series[idx_v] >= peak_val_at_dd:
        recovery_weeks = (idx_v - dd_end) * 4
        break
# Calmar ratio = CAGR / Max Drawdown
calmar = cagr / max_dd if max_dd > 0 else 0
# Sharpe ratio approssimato (rendimento settimanale / volatilita)
weekly_rets = []
for i in range(1, len(weekly_rows)):
    v_prev = weekly_rows[i-1]["valore"]
    v_curr = weekly_rows[i]["valore"]
    if v_prev > 0: weekly_rets.append((v_curr - v_prev) / v_prev)
import numpy as np
wr = np.array(weekly_rets)
sharpe = (wr.mean() / wr.std() * (52**0.5)) if wr.std() > 0 else 0
# Anni positivi / negativi
ann_pos = sum(1 for l in ann_rows if l["plp"] > 0)
ann_neg = sum(1 for l in ann_rows if l["plp"] <= 0)
print(f"\nBACKTEST REALE -- PAC v3.4.7")
print(f"="*50)
print(f"Capitale versato:   {capitale_ext:>12,.0f} EUR")
print(f"Valore finale:      {vf:>12,.0f} EUR")
print(f"P&L:                {pl_f:>+12,.0f} EUR ({plp:>+.1f}%)")
print(f"-"*50)
bollo_annuo = 0.002
tasse_pl    = pl_f * 0.26 if pl_f > 0 else 0
vf_netto    = vf - tasse_pl - (vf * bollo_annuo * n_years)
cagr_netto  = ((vf_netto/capitale_ext)**(1/n_years)-1)*100 if capitale_ext>0 else 0
print(f"CAGR nominale:      {cagr:>+12.2f}%/anno")
print(f"CAGR reale:         {reale:>+12.2f}%/anno")
print(f"CAGR netto*:        {cagr_netto:>+12.2f}%/anno  (*bollo 0.2% + tasse 26%)")
print(f"-"*50)
print(f"Max Drawdown:       {max_dd:>12.1f}%")
print(f"  Durata drawdown:  {dd_weeks:>9} settimane")
dd_date = log[dd_start]["date"] if dd_start < len(log) else "--"
dd_end_date = log[dd_end]["date"] if dd_end < len(log) else "--"
print(f"  Inizio:           {dd_date:>12}")
print(f"  Fondo:            {dd_end_date:>12}")
if recovery_weeks is not None:
    print(f"  Recupero:         {recovery_weeks:>9} settimane")
else:
    print(f"  Recupero:         non ancora avvenuto")
print(f"-"*50)
print(f"Calmar ratio:       {calmar:>12.2f}  (CAGR / MaxDD, > 1 = ottimo)")
print(f"Sharpe ratio:       {sharpe:>12.2f}  (> 1 = buono, > 2 = eccellente)")
print(f"Anni positivi:      {ann_pos:>12} / {len(ann_rows)}")
print(f"Anni negativi:      {ann_neg:>12} / {len(ann_rows)}")
print(f"-"*50)
print(f"Vendite pareggio:   {sum(1 for e in events if e["tipo"]=="PAREGGIO"):>12}")
print(f"Congelamenti:       {sum(1 for e in events if e["tipo"]=="CONGELATO"):>12}")
# Costruisci ann_rows PRIMA delle metriche (fix ordine)
seen=set(); ann_rows=[]
for l in log:
    if l["year"] not in seen: seen.add(l["year"]); ann_rows.append(l)
pd.DataFrame(weekly_rows).to_csv(PATHS["backtest_weekly"],index=False)
pd.DataFrame(move_rows).to_csv(PATHS["backtest_moves"],index=False)
pd.DataFrame(ann_rows).to_csv(PATHS["backtest_annual"],index=False)
dates_l=[l["date"] for l in log]; vals_l=[l["val"] for l in log]; invs_l=[l["inv"] for l in log]
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={"height_ratios":[3,1]})
# Grafico 1: Valore vs Versato
ax1 = axes[0]
ax1.plot(dates_l,vals_l,color="#1E5631",lw=2.5,label="Valore portafoglio")
ax1.plot(dates_l,invs_l,color="#AAAAAA",lw=1,ls="--",label="Capitale versato")
ax1.fill_between(dates_l,invs_l,vals_l,where=[v>i for v,i in zip(vals_l,invs_l)],alpha=0.15,color="#1E5631")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f"EUR {x:,.0f}"))
# Benchmark MSCI World (IWDA) per confronto
try:
    bm_raw = yf_download_safe("IWDA.AS", start=PAC_START_BT,
                              interval="1wk", auto_adjust=True)
    if bm_raw is not None and not bm_raw.empty:
        bm_close = bm_raw["Close"] if "Close" in bm_raw.columns \
                   else bm_raw["Close"]["IWDA.AS"]
        bm_close = bm_close.dropna()
                bm_start = float(bm_close.iloc[0])
        ts_idx   = pd.DatetimeIndex([pd.Timestamp(d) for d in dates_l])
        bm_ser   = bm_close.reindex(ts_idx, method="ffill").fillna(bm_start)
        bm_clean = [float(v)/bm_start*invs_l[i] for i,v in enumerate(bm_ser)]
        dates_bm = dates_l
        ax1.plot(dates_bm, bm_clean, color="#888800", lw=1.2,
                 ls="-.", alpha=0.7, label="MSCI World (IWDA)")
        bm_final = bm_clean[-1] if bm_clean else None
        bm_label = f" | MSCI World: EUR {bm_final:,.0f}" if bm_final else ""
    else:
        bm_label = ""
except Exception as e:
    logger.warning(f"Benchmark download fallito: {e}")
    bm_label = ""
ax1.set_title(f"PAC v3.4.7 | EUR {vf:,.0f} | CAGR {cagr:.2f}% | Reale {reale:.2f}% | MaxDD {max_dd:.1f}% | Calmar {calmar:.2f}{bm_label}",
              fontsize=11,fontweight="bold")
ax1.legend(); ax1.grid(alpha=0.3)
# Grafico 2: Drawdown nel tempo
ax2 = axes[1]
dd_series = []
peak_dd = vals_series[0]
for v in vals_series:
    if v > peak_dd: peak_dd = v
    dd_series.append(-(peak_dd - v) / peak_dd * 100 if peak_dd > 0 else 0)
ax2.fill_between(dates_l, dd_series, 0, alpha=0.6, color="#C00000")
ax2.axhline(0, color="black", lw=0.8)
ax2.axhline(-max_dd, color="#C00000", lw=1, ls="--", alpha=0.5, label=f"Max DD: -{max_dd:.1f}%")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f"{x:.0f}%"))
ax2.set_title("Drawdown nel tempo", fontsize=10)
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PATHS["backtest_chart"],dpi=150,bbox_inches="tight"); plt.show()
print("\nANNO PER ANNO:")
for l in ann_rows:
    t5=" - ".join(l["top5"][:3])
    print(f"{l['year']} EUR {l['inv']:>9,.0f} -> EUR {l['val']:>9,.0f}  {l['plp']:>+6.1f}%  {t5}")
print(f"\nOK: backtest_weekly.csv | backtest_movements.csv | backtest_annual.csv | backtest_reale.png")
